In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: ingestion_utils_notebook (parameterized)
# Purpose: Shared utility functions for all Bronze ingestion pipelines
# Features: Auto Loader + Schema hints + Critical DQ (quarantine) + Warn DQ (log)

import json
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit, col

# ============================================================
# load_config — load source-specific section from config JSON
# ============================================================
def load_config(config_path, source_key):
    with open(config_path, "r") as f:
        full_config = json.load(f)
    if source_key not in full_config:
        raise ValueError(f"Source '{source_key}' not found. Available: {list(full_config.keys())}")
    return full_config[source_key]


# ============================================================
# build_dq_conditions — return both critical and warn conditions
# ============================================================
def build_dq_conditions(dq_rules):
    """
    Returns:
        critical_condition: SQL string — rows that PASS all critical rules (use as valid filter)
        warn_conditions:    list of (rule_name, condition_str) — just for logging
    """
    critical = dq_rules.get("critical", [])
    warns = dq_rules.get("warn", [])
    
    critical_condition = " AND ".join([f"({r['condition']})" for r in critical]) if critical else "1=1"
    
    return critical_condition, [(r["name"], r["condition"]) for r in warns]


# ============================================================
# read_source_stream — unified Auto Loader read + metadata
# ============================================================
def read_source_stream(spark, cfg, source_name):
    print(f"📥 [{source_name}] Reading from: {cfg['source_path']}")
    print(f"   Target: {cfg['target_table']}")
    print(f"   Format: {cfg['format']}")
    
    reader = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", cfg["format"])
              .option("cloudFiles.schemaLocation", f"{cfg['checkpoint']}/schema")
              .option("cloudFiles.inferColumnTypes", "true")
              .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
              .option("rescuedDataColumn", "_rescued_data"))
    
    if cfg["format"] == "csv":
        csv_opts = cfg.get("csv_options", {"header": "true", "inferSchema": "true"})
        reader = reader.option("header", csv_opts.get("header", "true")) \
                       .option("inferSchema", csv_opts.get("inferSchema", "true"))
    
    if cfg.get("schema_hints"):
        reader = reader.option("cloudFiles.schemaHints", cfg["schema_hints"])
    
    df = reader.load(cfg["source_path"])
    
    df = (df
          .withColumn("_ingest_ts", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path")))
    
    return df


# ============================================================
# split_valid_and_quarantine — split stream into valid + quarantine
# ============================================================
def split_valid_and_quarantine(df, dq_rules, source_name):
    critical_condition, warn_conditions = build_dq_conditions(dq_rules)
    
    print(f"   Critical DQ (valid filter): {critical_condition}")
    for name, cond in warn_conditions:
        print(f"   Warn DQ (log only): {name} → {cond}")
    
    valid_df = df.filter(critical_condition)
    
    quarantine_df = (df.filter(f"NOT ({critical_condition})")
                       .withColumn("_dq_rule_failed", 
                                   F.concat_ws(", ",
                                       *[F.when(~F.expr(r["condition"]), F.lit(r["name"])) 
                                         for r in dq_rules.get("critical", [])])
                                   )
                       .withColumn("_quarantined_at", current_timestamp())
                       .withColumn("_source_name", lit(source_name)))
    
    return valid_df, quarantine_df


# ============================================================
# write_to_table — unified Delta stream writer
# ============================================================
def write_to_table(df, target_table, checkpoint_path, partition_col=None):
    writer = (df.writeStream
              .format("delta")
              .outputMode("append")
              .trigger(availableNow=True)
              .option("checkpointLocation", checkpoint_path)
              .option("mergeSchema", "true"))
    
    if partition_col:
        writer = writer.partitionBy(partition_col)
    
    writer.toTable(target_table)
    print(f"✅ Stream started → {target_table}")


# ============================================================
# full_bronze_ingestion — one-call wrapper for any source
# ============================================================
def full_bronze_ingestion(spark, cfg, source_name, partition_col=None):
    print(f"\n{'='*60}")
    print(f"🚀 Starting Bronze ingestion: {source_name}")
    print(f"{'='*60}")
    
    raw_df = read_source_stream(spark, cfg, source_name)
    
    valid_df, quarantine_df = split_valid_and_quarantine(raw_df, cfg["dq_rules"], source_name)
    
    write_to_table(valid_df, cfg["target_table"], 
                   f"{cfg['checkpoint']}/valid", partition_col)
    
    # 👉 env-aware: config_setup ke BRONZE se aata hai (hardcoded 'bronze.' nahi)
    quarantine_table = f"{BRONZE}._quarantine_{source_name}"
    write_to_table(quarantine_df, quarantine_table,
                   f"{cfg['checkpoint']}/quarantine")
    
    print(f"\n✅ [{source_name}] Bronze ingestion live:")
    print(f"   Valid      → {cfg['target_table']}")
    print(f"   Quarantine → {quarantine_table}")


print("✅ ingestion_utils_notebook loaded (parameterized):")
print("   • load_config(config_path, source_key)")
print("   • read_source_stream(spark, cfg, source_name)")
print("   • split_valid_and_quarantine(df, dq_rules, source_name)")
print("   • write_to_table(df, target_table, checkpoint_path, partition_col=None)")
print("   • full_bronze_ingestion(spark, cfg, source_name, partition_col=None)  ⭐")